<a href="https://colab.research.google.com/github/muneer-ahmad10/Chatbot/blob/main/RAG_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install streamlit pypdf sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 93.7 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader

def extract_text(pdf_file):

    reader = PdfReader(pdf_file)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    return text

In [ ]:
pdf_filename = "/content/Polymorphism.pdf"

pdf_text = extract_text(pdf_filename)

In [ ]:
print(pdf_text[:600])

In [ ]:
def create_chunks(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i+chunk_size]

        chunks.append(chunk)

    return chunks

In [ ]:
chunks = create_chunks(pdf_text)
print("Total Chunks:", len(chunks))

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(chunks)

In [ ]:
print(chunk_embeddings.shape)

In [ ]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

In [ ]:
print(index.ntotal)

In [ ]:
len(chunks)

In [ ]:
def retrieve_context(question):

    question_embedding = embedding_model.encode(
        [question]
    )

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        k=3
    )

    retrieved_chunks = []

    for idx in indices[0]:

        if idx != -1:

            retrieved_chunks.append(
                chunks[idx]
            )

    return "\n".join(retrieved_chunks)

In [ ]:
context = retrieve_context(
    "What is Polymorphism?"
)

print(context)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

In [ ]:
def answer_question(question):

    context = retrieve_context(question)

    prompt = f"""
    Answer the question using only
    the provided context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
question = "What is Polymorphism?"

answer = answer_question(
    question
)

print(answer)

# **Creating APP.py**

In [4]:
!pip install langchain-text-splitters

In [4]:
# %%writefile app.py

# import streamlit as st
# import numpy as np
# import faiss

# from pypdf import PdfReader

# from sentence_transformers import SentenceTransformer

# from transformers import (
#     AutoTokenizer,
#     AutoModelForSeq2SeqLM
# )



# st.set_page_config(page_title="PDF Chatbot")

# st.title("📚 PDF RAG Chatbot")

# st.write("Upload a PDF and ask questions.")



# @st.cache_resource
# def load_models():

#     embedding_model = SentenceTransformer(
#         "all-MiniLM-L6-v2"
#     )

#     tokenizer = AutoTokenizer.from_pretrained(
#         "google/flan-t5-base"
#     )

#     llm = AutoModelForSeq2SeqLM.from_pretrained(
#         "google/flan-t5-base"
#     )

#     return embedding_model, tokenizer, llm




# embedding_model, tokenizer, llm = load_models()



# # uploaded_file = st.file_uploader(
# #     "Upload PDF",
# #     type=["pdf"]
# # )

# uploaded_files = st.file_uploader(
#     "Upload PDFs",
#     type=["pdf"],
#     accept_multiple_files=True
# )



# def extract_text(pdf_file):

#     reader = PdfReader(pdf_file)

#     text = ""

#     for page in reader.pages:

#         page_text = page.extract_text()

#         if page_text:

#             text += page_text

#     return text


# def retrieve_context(question):

#     q_embedding = embedding_model.encode([question])

#     distances, indices = index.search(
#         np.array(q_embedding).astype("float32"),
#         k=3
#     )

#     retrieved_chunks = []

#     for idx in indices[0]:

#         if idx != -1:
#             retrieved_chunks.append(chunks[idx])

#     return retrieved_chunks, distances




# def answer_question(question):

#     retrieved_chunks, _ = retrieve_context(question)

#     context = "\n".join(retrieved_chunks)

#     prompt = f"""
#     Answer the question using only the provided context.

#     Context:
#     {context}

#     Question:
#     {question}

#     Answer:
#     """

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=512
#     )

#     outputs = llm.generate(
#         **inputs,
#         max_new_tokens=100
#     )

#     answer = tokenizer.decode(
#         outputs[0],
#         skip_special_tokens=True
#     )

#     return answer, retrieved_chunks
# # def create_chunks(text, chunk_size=500):

# #     chunks = []

# #     for i in range(0, len(text), chunk_size):

# #         chunks.append(
# #             text[i:i+chunk_size]
# #         )

# #     return chunks

# from langchain_text_splitters import RecursiveCharacterTextSplitter

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,
#     chunk_overlap=100
# )



# if uploaded_files:

#     all_text = ""

#     for pdf in uploaded_files:

#         all_text += extract_text(pdf) + "\n"

#     chunks = splitter.split_text(all_text)

#     chunk_embeddings = embedding_model.encode(chunks)

#     st.success(
#         f"Indexed {len(chunks)} chunks."
#     )




# dimension = chunk_embeddings.shape[1]

# index = faiss.IndexFlatL2(
#     dimension
# )

# index.add(
#     np.array(chunk_embeddings)
#     .astype("float32")
# )



# prompt = st.chat_input("Ask a question...")
# # question = st.text_input(
# #     "Ask a question"
# # )

# if "messages" not in st.session_state:

#     st.session_state.messages = []


# for message in st.session_state.messages:

#     with st.chat_message(message["role"]):

#         st.write(message["content"])
# # def retrieve_context(question):

# #     q_embedding = embedding_model.encode(
# #         [question]
# #     )

# #     distances, indices = index.search(
# #         np.array(q_embedding)
# #         .astype("float32"),
# #         k=3
# #     )

# #     retrieved = []

# #     for idx in indices[0]:

# #         if idx != -1:

# #             retrieved.append(
# #                 chunks[idx]
# #             )

# #     return "\n".join(retrieved)




# if prompt and uploaded_files:

#     with st.chat_message("user"):

#         st.write(prompt)

#     st.session_state.messages.append(
#         {
#             "role": "user",
#             "content": prompt
#         }
#     )

#     with st.spinner("Thinking..."):

#         answer, retrieved_chunks = answer_question(prompt)

#     with st.chat_message("assistant"):

#         st.write(answer)

#     st.session_state.messages.append(
#         {
#             "role": "assistant",
#             "content": answer
#         }
#     )



#     # st.subheader("Retrieved Sources")

#     # for i, chunk in enumerate(retrieved_chunks):

#     #     with st.expander(f"Source {i+1}"):

#     #         st.write(chunk)









# st.subheader("Retrieved Sources")

# for i, chunk in enumerate(retrieved_chunks):

#     with st.expander(f"Source {i+1}"):

#         st.write(chunk)










# # if "messages" not in st.session_state:

# #     st.session_state.messages = []



# for message in st.session_state.messages:

#     with st.chat_message(
#         message["role"]
#     ):
#         st.write(message["content"])



# if prompt:

#     st.session_state.messages.append(
#         {
#             "role":"user",
#             "content":prompt
#         }
#     )



# # answer = answer_question(prompt)
# answer, retrieved_chunks = answer_question(prompt)

# st.session_state.messages.append(
#     {
#         "role":"assistant",
#         "content":answer
#     }
# )


Overwriting app.py


In [ ]:
pdf_filename = "/content/file_1.pdf"

pdf_text = extract_text(pdf_filename)

In [5]:
%%writefile app.py
import streamlit as st
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_text_splitters import RecursiveCharacterTextSplitter

st.set_page_config(page_title="PDF Chatbot")
st.title("📚 PDF RAG Chatbot")
st.write("Upload a PDF and ask questions.")

# Load Models
@st.cache_resource
def load_models():
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    return embedding_model, tokenizer, llm

embedding_model, tokenizer, llm = load_models()

#Helper Functions
def extract_text(pdf_file):
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text
    return text

def answer_question(question, index, chunks):

    q_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(q_embedding).astype("float32"),
        k=3
    )

    retrieved_chunks = []
    retrieved_sources = []

    for idx in indices[0]:
        if idx != -1:
            retrieved_chunks.append(
                  all_chunks[idx]["text"]
              )

            retrieved_sources.append(
                  all_chunks[idx]["source"]
              )

    context = "\n".join(retrieved_chunks)


    chat_history = ""

    for msg in st.session_state.messages[-4:]:

        chat_history += (
            f"{msg['role']}: {msg['content']}\n"
        )

    llm_prompt = f"""
    You are a helpful assistant.

    Use the conversation history and context.

    Conversation History:
    {chat_history}

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        llm_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks,retrieved_sources,distances


with st.sidebar:

    st.header("Project Info")

    st.write(
        """
        PDF RAG Chatbot

        - Sentence Transformers
        - FAISS
        - FLAN-T5
        - Streamlit
        """
    )



#Initialize Session State for Chat History
if "messages" not in st.session_state:
    st.session_state.messages = []

# Sidebar / Upload Section
uploaded_files = st.file_uploader("Upload PDFs", type=["pdf"], accept_multiple_files=True)

#Process PDFs and Build FAISS Index if files are present
index = None
chunks = []

if uploaded_files:
    # all_text = ""
    all_chunks = []

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

    for pdf in uploaded_files:

        pdf_text = extract_text(pdf)

        pdf_chunks = splitter.split_text(pdf_text)

        for chunk in pdf_chunks:

            all_chunks.append(
                {
                    "text": chunk,
                    "source": pdf.name
                }
            )
    chunks = [
    item["text"]
    for item in all_chunks
     ]

    # chunks = splitter.split_text(all_text)

    if chunks:
        chunk_embeddings = embedding_model.encode(chunks,batch_size=32,show_progress_bar=True)
        dimension = chunk_embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(np.array(chunk_embeddings).astype("float32"))
        st.success(f"Indexed {len(chunks)} chunks.")

#Display Chat History
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])
        if "sources" in message:
            for i, chunk in enumerate(message["sources"]):
                with st.expander(f"Source {i+1} (Retrieved Chunk)"):
                    st.write(chunk)

# Handle New User Input

prompt = st.chat_input("Ask a question...")
if prompt:
    if not uploaded_files or index is None:
        st.error("Please upload and process a PDF first!")
    else:
        # Display user message
        with st.chat_message("user"):
            st.write(prompt)
        st.session_state.messages.append({"role": "user", "content": prompt})

        # Generate answer with spinner
        with st.spinner("Thinking..."):
            answer,retrieved_chunks,retrieved_sources,distances = answer_question(prompt, index, chunks)
        with st.expander("Similarity Scores"):
            for i, score in enumerate(distances[0]):
                st.write(
                      f"Chunk {i+1}: {round(float(score),4)}"
                    )

        # Display assistant message
        with st.chat_message("assistant"):
            st.write(answer)
            for i, chunk in enumerate(retrieved_chunks):

                with st.expander(
                    f"Source {i+1} - {retrieved_sources[i]}"
                ):

                    st.write(chunk)


        st.session_state.messages.append({
            "role": "assistant",
            "content": answer,
            "sources": retrieved_chunks
        })

Overwriting app.py


In [9]:
!pip install streamlit pyngrok
!streamlit run app.py &>/content/logs.txt &

from pyngrok import ngrok

ngrok.set_auth_token("3DQMvg7DN2UTMoMvgHFxvhcT66M_7v4cXJhmhSmNNaXvK447J")

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://handbook-marry-hazy.ngrok-free.dev" -> "http://localhost:8501"
